In [0]:
# ==============================================================================
# DOCUMENTAÇÃO DO NOTEBOOK
# Nome: 00_dicionario_de_dados_gold
# Camada: Gold (Documentação e Metadados)
# Descrição: 
#   Este notebook aplica comentários descritivos e regras de negócio nas colunas 
#   e na tabela da camada Gold (gold_kpis_diarios) dentro do catálogo do Databricks, 
#   garantindo governança e clareza para consumo analítico e relatórios.
# ==============================================================================

from pathlib import Path
from pyspark.sql.functions import current_timestamp, lit

catalogo = "will"
esquema = "i30"

def sql_texto(texto):
    return "'" + texto.replace("'", "''") + "'"

comentarios_gold = {
    "gold_kpis_diarios": {
        "tabela": "Camada Gold consolidada contendo os KPIs diários, cruzando viagens, caronas e abastecimentos para análise de desempenho, custos e lucratividade do veículo.",
        "colunas": {
            "DATA": "Data de referência consolidada. Regra de negócio: Chave de junção primária (Join) unificando as informações diárias de todas as fontes.",
            "KM_RODADAS": "Quilometragem total percorrida no dia, em km. Regra de negócio: Herdada da camada Silver de viagens (KM_FINAL - KM_INICIAL).",
            "PAINEL_L_100KM": "Consumo médio indicado pelo painel do veículo, em litros por 100 km. Regra de negócio: Dado analítico de eficiência obtido da tabela de viagens.",
            "VELOC_MEDIA_KM_H": "Velocidade média desenvolvida no dia, em km/h. Regra de negócio: Indicador de desempenho de tráfego herdado da Silver.",
            "TOTAL_PASSAGEIROS": "Quantidade total de passageiros transportados no dia. Regra de negócio: Soma dos passageiros da manhã e da noite (PASSAG_MANHA + PASSAG_NOITE), tratando nulos como zero.",
            "TOTAL_GANHO_REAIS": "Faturamento total obtido com caronas no dia, em reais (R$). Regra de negócio: Soma dos ganhos da manhã e da noite (GANHO_MANHA_REAIS + GANHO_NOITE_REAIS).",
            "CUSTO_ABASTECIMENTO_REAIS": "Gasto total com combustível no dia, em reais (R$). Regra de negócio: Valor total pago nos abastecimentos do dia (POSTO_VALOR_TOTAL), tratando dias sem abastecimento como zero.",
            "LITROS_ABASTECIDOS": "Volume total de combustível abastecido no dia, em litros. Regra de negócio: Quantidade de litros colocada no tanque, tratando dias sem abastecimento como zero.",
            "LUCRO_LIQUIDO_DIA": "Resultado financeiro líquido diário, em reais (R$). Regra de negócio: Enriquecimento de KPI calculado subtraindo o custo de abastecimento do ganho total de caronas (TOTAL_GANHO_REAIS - CUSTO_ABASTECIMENTO_REAIS).",
            "CUSTO_POR_KM": "Custo financeiro por quilômetro rodado, em reais por km (R$/km). Regra de negócio: KPI calculado dividindo o custo de abastecimento pelas quilometragens rodadas quando KM_RODADAS > 0 (CUSTO_ABASTECIMENTO_REAIS / KM_RODADAS), caso contrário 0.0."
        },
    }
}

for tabela, definicao in comentarios_gold.items():
    tabela_completa = f"`{catalogo}`.`{esquema}`.`{tabela}`"

    # Aplica comentário na tabela
    spark.sql(
        f"COMMENT ON TABLE {tabela_completa} "
        f"IS {sql_texto(definicao['tabela'])}"
    )

    # Aplica comentários nas colunas
    for coluna, comentario in definicao["colunas"].items():
        coluna_sql = f"`{coluna.replace('`', '``')}`"

        spark.sql(
            f"COMMENT ON COLUMN {tabela_completa}.{coluna_sql} "
            f"IS {sql_texto(comentario)}"
        )

    print(f"Dicionário Gold aplicado com sucesso: {catalogo}.{esquema}.{tabela}")